# Generation for Charging Stations depending on relative Path Occupation

## Research Question

The feasibility of intelligently placing electrical charging stations on highly visited roads. SUMO and MatSIM deliver a full framework for simulating realistic traffic.


## Methods to determine traffic

- Occupation of lanes
- Color sections by own criteria, depending on traffic (manually, or by automation script) 
- Occupation of certain areas

## Deciding on Nomenclature

### Calculate with Occupation
ex: Lane A is occupied by 2 cars, X with 5s (seconds) and Y with 10s, in a timeframe of 60s. Car X enters on second 20-25, car Y enters on 23-33.
The occupancy of Lane A is a total of 13 seconds of 60. 
$$ ( 13 / 60 = 0,2167 ) $$

Mathematically, the lane is occupied 21.67% of the time.  

### Calculate with Density    
ex: Lane A (500m) is populated with 10 cars.
$$ 10 (cars) / 0.5 (km) =  20  cars / km $$ 

With lots of traffic the value may approach a value of around 100 cars/km.

## Chosen method to determine traffic

The problem requires a different solution than the occupation of a lane, because a high occupation does not necessarily hint towards a lot of traffic. So, the initial method will be utilizing the density of each lane across the whole simulation time span.

Density allows to track every lane independently and extract information for these lanes. For now Occupation is less of a prioritiy.

## Design of method

The first step to complete the goal of finding a way to calculate beneficial positions for charging stations, is to extract the density and the lane's id, to which it belongs. These two values do not solve the problem itself, but deliver the information needed for setting up the criteria and a fitting algorithm.  


## Implementation of method

Below is the first part of code, that allows the extraction of the density and lane id into two variables.

In [157]:
import xml.etree.ElementTree as ET

tree = ET.parse('welzheim_edges.xml')
root = tree.getroot()

density_values = []
lane_ids = []
for edge in root.findall('.//edge'):
    density = edge.get('density')
    id = edge.get('id')
    if density:
        density_values.append(density)
        lane_ids.append(id)
        print("density (lane_id " + id + "): " + str(density))


density (lane_id 1): 0.02
density (lane_id 10): 0.00
density (lane_id 100): 0.12
density (lane_id 1000): 0.02
density (lane_id 1001): 0.03
density (lane_id 1002): 0.02
density (lane_id 1003): 0.02
density (lane_id 1004): 0.02
density (lane_id 1005): 0.02
density (lane_id 1006): 0.01
density (lane_id 1007): 0.02
density (lane_id 1008): 0.01
density (lane_id 1009): 0.01
density (lane_id 101): 0.07
density (lane_id 1010): 0.13
density (lane_id 1011): 0.08
density (lane_id 1012): 0.00
density (lane_id 1013): 0.01
density (lane_id 1014): 0.02
density (lane_id 1015): 0.02
density (lane_id 1016): 0.08
density (lane_id 1017): 0.06
density (lane_id 1018): 0.13
density (lane_id 1019): 0.13
density (lane_id 102): 0.13
density (lane_id 1020): 0.06
density (lane_id 1021): 0.03
density (lane_id 1022): 0.04
density (lane_id 1023): 0.40
density (lane_id 1024): 0.25
density (lane_id 1025): 0.10
density (lane_id 1026): 0.09
density (lane_id 1027): 0.03
density (lane_id 1028): 0.02
density (lane_id 1029)

## Algorithm to determine positioning of charging stations

The algorithm will take into account which lanes are busy and will put electrical charging stations depending on their traffic density.

## Requirements Engineering

It does not make sense to put charging stations everywhere, where there is a density value higher than X, assuming some streets might be overcrowded and others void of charging stations. So, a few criterias need to be elaborated to guarantee and efficient system to distribute charging stations.

## Modify sumocfg file

For the sumocfg to be able to handle additional information, a tag with "additional-files" needs to be added.

In [158]:
sumo_cfg = 'welzheim.sumocfg'
network_cfg = ET.parse(sumo_cfg)
root_cfg = network_cfg.getroot()

add = ""
for root in root_cfg.findall('.//additional-files'):
    add = root

if 'additional-files' in add.tag:
    print("additional-files already exists")
else:
    print("adding additional-files")
    additional = ET.Element('additional-files')
    additional.set('value', 'welzheim_edges.xml')
    root_cfg.append(additional)

network_cfg.write(sumo_cfg, encoding='utf-8', xml_declaration=True)


additional-files already exists


## Create the additional-files XML

The essential file for any further additions, like charging stations, adjustments for cars etc.

In [159]:
import os

welzheim_add = "welzheim.add.xml"

if not os.path.exists(welzheim_add):
    xml_content = """<?xml version="1.0" encoding="UTF-8"?>\n<additional>\n</additional>"""
    with open(welzheim_add, "w", encoding="utf-8") as file:
        file.write(xml_content)

## Algorithm to add charging stations

In [163]:
additional_tree = ET.parse(welzheim_add)
additional_root = additional_tree.getroot()

addition = root.find('.//additional')

for cs in additional_root.findall('.//chargingStation'):
    additional_root.remove(cs)

for density_values, lane_ids in zip(density_values, lane_ids):
    cs = ET.SubElement(additional_root, 'chargingStation')
    cs.set('id', 'cs_'+lane_ids)
    cs.set('lane', lane_ids)
    cs.set('startPos', '0')
    cs.set('endPos', '0')
    additional_root.append(cs)

additional_tree.write(welzheim_add, encoding='utf-8', xml_declaration=True)